In [2]:
import duckdb
from pathlib import Path

raw_dir = Path('../data/raw')
parquet_pattern = str(raw_dir / 'fhvhv_tripdata_2024-*.parquet')
zone_file = str(raw_dir / 'taxi_zone_lookup.csv')

con = duckdb.connect()

print(f"Pattern file yang akan diaudit: {parquet_pattern}")

Pattern file yang akan diaudit: ..\data\raw\fhvhv_tripdata_2024-*.parquet


# 1. Schema Consistency Across 12 Months

In [3]:
df_schema_all = con.execute(f"""
    SELECT file_name, name AS column_name, type AS column_type
    FROM parquet_schema('{parquet_pattern}')
""").df()

df_schema_all.head(10)

,file_name,column_name,column_type
0,..\data\raw\fhvhv_tripdata_2024-01.parquet,schema,NaN
1,..\data\raw\fhvhv_tripdata_2024-01.parquet,hvfhs_license_num,BYTE_ARRAY
2,..\data\raw\fhvhv_tripdata_2024-01.parquet,dispatching_base_num,BYTE_ARRAY
3,..\data\raw\fhvhv_tripdata_2024-01.parquet,originating_base_num,BYTE_ARRAY
4,..\data\raw\fhvhv_tripdata_2024-01.parquet,request_datetime,INT64
5,..\data\raw\fhvhv_tripdata_2024-01.parquet,on_scene_datetime,INT64
6,..\data\raw\fhvhv_tripdata_2024-01.parquet,pickup_datetime,INT64
7,..\data\raw\fhvhv_tripdata_2024-01.parquet,dropoff_datetime,INT64
8,..\data\raw\fhvhv_tripdata_2024-01.parquet,PULocationID,INT32
9,..\data\raw\fhvhv_tripdata_2024-01.parquet,DOLocationID,INT32


In [4]:
df_columns_only = df_schema_all[df_schema_all['column_type'].notna()]

col_count_per_file = df_columns_only.groupby('file_name').size()

col_count_per_file

file_name
..\data\raw\fhvhv_tripdata_2024-01.parquet    24
..\data\raw\fhvhv_tripdata_2024-02.parquet    24
..\data\raw\fhvhv_tripdata_2024-03.parquet    24
..\data\raw\fhvhv_tripdata_2024-04.parquet    24
..\data\raw\fhvhv_tripdata_2024-05.parquet    24
..\data\raw\fhvhv_tripdata_2024-06.parquet    24
..\data\raw\fhvhv_tripdata_2024-07.parquet    24
..\data\raw\fhvhv_tripdata_2024-08.parquet    24
..\data\raw\fhvhv_tripdata_2024-09.parquet    24
..\data\raw\fhvhv_tripdata_2024-10.parquet    24
..\data\raw\fhvhv_tripdata_2024-11.parquet    24
..\data\raw\fhvhv_tripdata_2024-12.parquet    24
dtype: int64

In [5]:
df_columns_only['col_signature'] = df_columns_only['column_name'] + '|' + df_columns_only['column_type']

signature_per_file = df_columns_only.groupby('file_name')['col_signature'].apply(set)

signature_jan = signature_per_file.iloc[0]

for file_name, sig in signature_per_file.items():
    is_same = (sig == signature_jan)
    print(f"{file_name}: {'SAMA' if is_same else 'BEDA!'}")

..\data\raw\fhvhv_tripdata_2024-01.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-02.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-03.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-04.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-05.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-06.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-07.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-08.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-09.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-10.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-11.parquet: SAMA
..\data\raw\fhvhv_tripdata_2024-12.parquet: SAMA


# 2. Referential & Timestamp Validity

In [6]:
df_timestamp_check = con.execute(f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE dropoff_datetime <= pickup_datetime) AS invalid_timestamp_count
    FROM read_parquet('{parquet_pattern}')
""").df()

df_timestamp_check

,total_rows,invalid_timestamp_count
0,239470448,9680


In [7]:
df_zone_check = con.execute(f"""
    SELECT 
        COUNT(*) FILTER (WHERE zone_pu.LocationID IS NULL) AS invalid_pickup_zone,
        COUNT(*) FILTER (WHERE zone_do.LocationID IS NULL) AS invalid_dropoff_zone
    FROM read_parquet('{parquet_pattern}') AS trip
    LEFT JOIN read_csv_auto('{zone_file}') AS zone_pu
        ON trip.PULocationID = zone_pu.LocationID
    LEFT JOIN read_csv_auto('{zone_file}') AS zone_do
        ON trip.DOLocationID = zone_do.LocationID
""").df()

df_zone_check

,invalid_pickup_zone,invalid_dropoff_zone
0,0,0


# 3. Distribution, Outliers & Completeness

In [8]:
df_miles_check = con.execute(f"""
    SELECT 
        MIN(trip_miles) AS min_miles,
        APPROX_QUANTILE(trip_miles, 0.25) AS q1_miles,
        APPROX_QUANTILE(trip_miles, 0.50) AS median_miles,
        APPROX_QUANTILE(trip_miles, 0.75) AS q3_miles,
        APPROX_QUANTILE(trip_miles, 0.99) AS p99_miles,
        MAX(trip_miles) AS max_miles,
        COUNT(*) FILTER (WHERE trip_miles <= 0) AS zero_or_negative_miles
    FROM read_parquet('{parquet_pattern}')
""").df()

df_miles_check

,min_miles,q1_miles,median_miles,q3_miles,p99_miles,max_miles,zero_or_negative_miles
0,0.0,1.571927,3.013404,6.377656,27.257162,555.25,34059


In [9]:
df_trip_check = con.execute(f"""
    SELECT 
        MIN(trip_time) AS min_time,
        APPROX_QUANTILE(trip_time, 0.25) AS q1_time,
        APPROX_QUANTILE(trip_time, 0.50) AS median_time,
        APPROX_QUANTILE(trip_time, 0.75) AS q3_time,
        APPROX_QUANTILE(trip_time, 0.99) AS p99_time,
        MAX(trip_time) AS max_time,
        COUNT(*) FILTER (WHERE trip_time <= 0) AS zero_or_negative_time
    FROM read_parquet('{parquet_pattern}')
""").df()

df_trip_check

,min_time,q1_time,median_time,q3_time,p99_time,max_time,zero_or_negative_time
0,0,604,978,1555,4296,55138,30


In [11]:
df_completeness_check = con.execute(f"""
    SELECT 
        COUNT(*) FILTER (WHERE trip.hvfhs_license_num IS NULL) AS null_hvfhs_license_num,
        COUNT(*) FILTER (WHERE trip.dispatching_base_num IS NULL) AS null_dispatching_base_num
    FROM read_parquet('{parquet_pattern}') AS trip
""").df()

df_completeness_check

,null_hvfhs_license_num,null_dispatching_base_num
0,0,0


# 4. Duplicate Detection

In [12]:
df_duplicate_check = con.execute(f"""
    WITH duplicate_group AS (
        SELECT
            dispatching_base_num,
            pickup_datetime,
            dropoff_datetime,
            PULocationID,
            DOLocationID,
            COUNT(*) AS duplicate_count
        FROM read_parquet('{parquet_pattern}')
        GROUP BY
            dispatching_base_num,
            pickup_datetime,
            dropoff_datetime,
            PULocationID,
            DOLocationID
            HAVING COUNT(*) > 1
        )
        SELECT
            COUNT(*) AS double_combination_total,
            SUM(duplicate_count) AS duplicate_total_rows
        FROM duplicate_group
""").df()

df_duplicate_check

,double_combination_total,duplicate_total_rows
0,439,879.0
